In [ ]:
%pip install -q \
"cohere>=5.13"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 352.0/352.0 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 76.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import random
import os
import cohere
import getpass
import pickle

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
PROJECT_PATH = "/content/drive/MyDrive/KeepCoding/Ai-Engineering/Practica"

DATA_PATH = os.path.join(PROJECT_PATH, "data")
os.makedirs(DATA_PATH, exist_ok=True)

In [ ]:
csv_path =os.path.join(DATA_PATH, "events_dataset.csv")

In [ ]:
df_events= pd.read_csv(csv_path)

Pasar cada fila del dataframe a una lista para que luego se pueda transformar datos estructurados en documentos que luego puedan convertirse en embeddings.

In [ ]:
documents=[]

for index, row in df_events.iterrows():
  texto = f"""
  Evento: {row['event_type']}

  Ciudad: {row['city']}

  Tipo de recinto: {row['venue_type']}

  Género musical: {row['music_genre']}

  Público objetivo: {row['target_audience']}

  Día de la semana: {row['day_of_week']}

  Hora de inicio: {row['start_time']}

  Precio de entrada: {row['ticket_price']} €

  Aforo: {row['capacity']} personas

  Ocupación: {row['occupancy_rate']:.0%}

  Inicio de ventas: {row['sales_start_days_before']} días antes

  Estrategia de promoción: {row['promotion_strategy']}

  Canal principal de venta: {row['main_sales_channel']}

  Copas incluidas: {row['drinks_included']}

  Climatología: {row['weather']}

  Eventos competidores: {row['competing_events']}

  Resultado del evento: {row['result']}

  Observaciones comerciales: {row['business_notes']}
  """
  documents.append(texto)

In [ ]:
documents_path = os.path.join(
    DATA_PATH,
    "event_documents.pkl"
)
with open(documents_path, "wb") as f:
    pickle.dump(documents, f)

No he añadido event_id porque para los embeddings no va a tener mucha relevancia al igual que tickets vendidos por que ya tengo porcentaje de ocupacion y capacidad

In [ ]:
print(len(documents))

2500


In [ ]:
print(documents[0])


  Evento: Fiesta reggaeton

  Ciudad: Málaga

  Tipo de recinto: Terraza

  Género musical: Techno

  Público objetivo: Erasmus

  Día de la semana: Viernes

  Hora de inicio: 00:00

  Precio de entrada: 12 €

  Aforo: 1200 personas

  Ocupación: 70%

  Inicio de ventas: 7 días antes

  Estrategia de promoción: Promoción grupos

  Canal principal de venta: Fourvenues

  Copas incluidas: 2

  Climatología: Calor

  Eventos competidores: 0

  Resultado del evento: Buen resultado

  Observaciones comerciales: Promoción por grupos funcionó muy bien
  


In [ ]:
if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key: ")

co = cohere.Client(os.environ["COHERE_API_KEY"])

Cohere API Key: ··········


Primero hago una prueba para ver que está todo bien

In [ ]:
documents_test = documents[:3]

In [ ]:
res = co.embed(
    texts=documents_test,
    model="embed-multilingual-v3.0",
    input_type="search_query",
    embedding_types=["float"],
)
embs = res.embeddings.float

In [ ]:
len(embs[0])

1024

In [ ]:
print(embs[0])

[0.002609253, 0.0065307617, 0.022369385, -0.0036296844, -0.01626587, 0.024475098, -0.0030078888, -0.04345703, -0.018951416, -0.050445557, 0.034942627, -0.028747559, -0.019439697, -0.010505676, -0.0284729, -0.00605011, 0.0039711, -0.028930664, -0.025299072, -0.004611969, -0.016952515, 0.04623413, 0.03161621, -0.009681702, 0.058807373, -0.029190063, -0.005115509, 0.053741455, 0.03363037, -0.05203247, 0.00472641, -0.005706787, 0.014877319, -0.011199951, -0.0065994263, -0.017440796, -0.0101623535, -0.083740234, -0.008491516, 0.071899414, -0.026321411, 0.024307251, 0.021484375, 0.01675415, -0.0061454773, 0.016555786, -0.01626587, -0.029571533, 0.01776123, 0.014862061, 0.011940002, 0.010910034, 0.005138397, -0.0064849854, 0.011993408, 0.033599854, 0.043548584, 0.015068054, -0.029159546, -0.027770996, 0.0056610107, -0.018081665, -0.003610611, 0.04171753, 0.013496399, 0.010108948, -0.013000488, 0.068603516, -0.002565384, 0.022201538, 0.031219482, 0.03463745, 0.025436401, 0.010475159, 0.0027770

Al ver que funciona todo correctamente ejecutamos todos los documentos

Proibé a ejecutarlo pero para la prueba gratuita excedía por lo que lo hice por bloques y utilicé search_document ya que es apra almacenar el documento
- "search_document": Used for embeddings stored in a vector database for search use-cases.
- "search_query": Used for embeddings of search queries run against a vector DB to find relevant documents.

In [ ]:
import time

all_embeddings = []

batch_size = 50

for i in range(0, len(documents), batch_size):
    batch = documents[i:i + batch_size]

    res = co.embed(
        texts=batch,
        model="embed-multilingual-v3.0",
        input_type="search_document",
        embedding_types=["float"],
    )

    all_embeddings.extend(res.embeddings.float)

    print(f"Procesados {i + len(batch)} / {len(documents)}")

    time.sleep(10)

Procesados 50 / 2500
Procesados 100 / 2500
Procesados 150 / 2500
Procesados 200 / 2500
Procesados 250 / 2500
Procesados 300 / 2500
Procesados 350 / 2500
Procesados 400 / 2500
Procesados 450 / 2500
Procesados 500 / 2500
Procesados 550 / 2500
Procesados 600 / 2500
Procesados 650 / 2500
Procesados 700 / 2500
Procesados 750 / 2500
Procesados 800 / 2500
Procesados 850 / 2500
Procesados 900 / 2500
Procesados 950 / 2500
Procesados 1000 / 2500
Procesados 1050 / 2500
Procesados 1100 / 2500
Procesados 1150 / 2500
Procesados 1200 / 2500
Procesados 1250 / 2500
Procesados 1300 / 2500
Procesados 1350 / 2500
Procesados 1400 / 2500
Procesados 1450 / 2500
Procesados 1500 / 2500
Procesados 1550 / 2500
Procesados 1600 / 2500
Procesados 1650 / 2500
Procesados 1700 / 2500
Procesados 1750 / 2500
Procesados 1800 / 2500
Procesados 1850 / 2500
Procesados 1900 / 2500
Procesados 1950 / 2500
Procesados 2000 / 2500
Procesados 2050 / 2500
Procesados 2100 / 2500
Procesados 2150 / 2500
Procesados 2200 / 2500
Procesad

In [ ]:
len(all_embeddings)

2500

In [ ]:
len(all_embeddings[0])

1024

In [ ]:
print(all_embeddings[0])

[-0.0026226044, 0.013748169, 0.04626465, -0.0005927086, -0.025360107, 0.02154541, -0.012611389, -0.0309906, -0.021148682, -0.034210205, 0.022445679, -0.017150879, -0.017349243, -0.028945923, -0.02709961, -0.020996094, -0.013824463, -0.032196045, -0.041229248, -0.0030269623, -0.0025310516, 0.04751587, 0.01247406, -0.018478394, 0.047058105, -0.026382446, -0.015151978, 0.06402588, 0.02508545, -0.057800293, 0.013748169, 0.0013856888, 0.0013723373, -0.00844574, -0.0065841675, -0.004886627, -0.0023460388, -0.06506348, -0.001868248, 0.0546875, -0.016525269, 0.017028809, 0.014533997, 0.0041389465, -0.006198883, 0.047088623, -0.022216797, -0.05496216, 0.044891357, 0.009109497, 0.014266968, 0.03869629, -0.008590698, -0.013442993, -0.017181396, 0.034210205, 0.015731812, 0.0030326843, 0.015289307, -0.027511597, 0.008079529, -0.0070152283, 0.017913818, 0.018035889, -0.025527954, -0.005908966, -0.010475159, 0.08416748, 0.00687027, 0.029037476, 0.02418518, 0.055267334, 0.01637268, 0.015625, -0.004917

Me he ayudado de la IA para saber como guardar los embeddings

In [ ]:
embeddings_path = os.path.join(
    DATA_PATH,
    "event_embeddings.pkl"
)

with open(embeddings_path, "wb") as f:
    pickle.dump(all_embeddings, f)